In [ ]:
import pandas as pd
import numpy as np

In [ ]:
print("="*60)
print("MIMIC-III")
print("="*60)

admissions = pd.read_csv('mimic-iii/ADMISSIONS.csv')
diagnoses_icd9 = pd.read_csv('mimic-iii/DIAGNOSES_ICD.csv')

print("Admissions shape:", admissions.shape)
print("Diagnoses shape:", diagnoses_icd9.shape)
print("\nAdmissions columns:", admissions.columns.tolist())
print("Diagnoses columns:", diagnoses_icd9.columns.tolist())

# 从admissions数据中计算每个病人的住院次数
patient_admission_counts = admissions.groupby('SUBJECT_ID')['HADM_ID'].nunique()

print(f"\nNumber of patients: {len(patient_admission_counts)}")
print(f"Average number of admissions per patient: {patient_admission_counts.mean():.2f}")
print(f"Maximum number of admissions per patient: {patient_admission_counts.max()}")
print(f"Minimum number of admissions per patient: {patient_admission_counts.min()}")

In [ ]:
print("="*60)
print("MIMIC-IV")
print("="*60)

admissions_mimic4 = pd.read_csv('mimic-iv/admissions.csv.gz')
diagnoses_mimic4 = pd.read_csv('mimic-iv/diagnoses_icd.csv.gz')

print("Admissions shape:", admissions_mimic4.shape)
print("Diagnoses shape:", diagnoses_mimic4.shape)
print("\nAdmissions columns:", admissions_mimic4.columns.tolist())
print("Diagnoses columns:", diagnoses_mimic4.columns.tolist())

# 从admissions数据中计算每个病人的住院次数
patient_admission_counts = admissions_mimic4.groupby('subject_id')['hadm_id'].nunique()

print(f"\nNumber of patients: {len(patient_admission_counts)}")
print(f"Average number of admissions per patient: {patient_admission_counts.mean():.2f}")
print(f"Maximum number of admissions per patient: {patient_admission_counts.max()}")
print(f"Minimum number of admissions per patient: {patient_admission_counts.min()}")

In [ ]:
def preprocess_data(diagnoses, admissions):
    """Preprocess the MIMIC-III data"""
    
    diagnoses_clean = diagnoses.copy()
    diagnoses_clean = diagnoses_clean.dropna(subset=['SUBJECT_ID', 'HADM_ID', 'ICD9_CODE'])
    
    admissions_clean = admissions.copy()
    admissions_clean = admissions_clean.dropna(subset=['SUBJECT_ID', 'HADM_ID', 'ADMITTIME'])
    
    # time format conversion
    admissions_clean['ADMITTIME'] = pd.to_datetime(admissions_clean['ADMITTIME'])
    admissions_clean['DISCHTIME'] = pd.to_datetime(admissions_clean['DISCHTIME'])
    
    print(f"Number of Diagnoses: {len(diagnoses_clean)}")
    print(f"Number of Admissions: {len(admissions_clean)}")
    print(f"The number of unique patients: {diagnoses_clean['SUBJECT_ID'].nunique()}")
    print(f"The number of unique admissions: {diagnoses_clean['HADM_ID'].nunique()}")
    
    return diagnoses_clean, admissions_clean

diagnoses_clean, admissions_clean = preprocess_data(diagnoses_icd9, admissions)

In [ ]:
# 全集
unique_icd9 = diagnoses_clean['ICD9_CODE'].nunique()
print(f"Number of unique ICD-9 codes: {unique_icd9}")

unique_pairs = unique_icd9 * (unique_icd9 - 1) // 2
print(f"Number of unique ICD-9 code pairs: {unique_pairs}")   # 24M unique pairs (whole set)

In [ ]:
merged_data = diagnoses_clean.merge(
    admissions_clean[['SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME']], 
    on=['SUBJECT_ID', 'HADM_ID'], 
    how='inner'
)
# 按病人和入院时间排序
merged_data = merged_data.sort_values(['SUBJECT_ID', 'ADMITTIME', 'SEQ_NUM'])
merged_data.head(10)

In [ ]:
# 目前默认min_visits=1，即每个病人至少有一次入院记录
def extract_patient_trajectories(diagnoses_df, admissions_df, min_visits=1):
    """
    Extract patient trajectories: a sequence of visits with ICD9 codes per patient
    
    Args:
        diagnoses_df: DataFrame, diagnoses data
        admissions_df: DataFrame, admissions data
        min_visits: int, minimum number of visits
        
    Returns:
        patient_trajectories: dict, key is subject_id, value is a list of trajectories
        patient_info: dict, contains information of patients
    """
    
    # Merge diagnoses with admissions
    merged_data = diagnoses_df.merge(
        admissions_df[['SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME']], 
        on=['SUBJECT_ID', 'HADM_ID'], 
        how='inner'
    )
    
    merged_data = merged_data.sort_values(['SUBJECT_ID', 'ADMITTIME', 'SEQ_NUM'])

    patient_trajectories = {}
    patient_info = {}
    
    for subject_id in merged_data['SUBJECT_ID'].unique():
        patient_data = merged_data[merged_data['SUBJECT_ID'] == subject_id]

        admissions = patient_data.groupby('HADM_ID')

        # if len(admissions) < min_visits or len(admissions) > 5:
        if len(admissions) < min_visits:
            continue

        trajectory = []
        admission_dates = []
        discharge_dates = []

        for hadm_id, admission_data in admissions:
            # 按SEQ_NUM排序获取该次入院的诊断代码
            codes = admission_data.sort_values('SEQ_NUM')['ICD9_CODE'].tolist()
            admission_date = admission_data['ADMITTIME'].iloc[0]
            discharge_date = admission_data['DISCHTIME'].iloc[0]
            
            trajectory.append(codes)
            admission_dates.append(admission_date)
            discharge_dates.append(discharge_date)
        patient_trajectories[subject_id] = trajectory
        patient_info[subject_id] = {
            'num_admissions': len(admissions),
            'admission_dates': admission_dates,
            'discharge_dates': discharge_dates,
            'total_diagnoses': sum(len(visit) for visit in trajectory)
        }

    return patient_trajectories, patient_info

In [ ]:
patient_trajectories, patient_info = extract_patient_trajectories(diagnoses_clean, admissions_clean)

print(f"Successfully extracted {len(patient_trajectories)} patient trajectories")
print(f"\nAverage number of admissions per patient: {np.mean([info['num_admissions'] for info in patient_info.values()]):.2f}")
print(f"Maximum number of admissions per patient: {np.max([info['num_admissions'] for info in patient_info.values()])}")
print(f"Average number of diagnoses per patient: {np.mean([info['total_diagnoses'] for info in patient_info.values()]):.2f}")
print(f"Average number of diagnoses per admission: {np.mean([info['total_diagnoses'] / info['num_admissions'] for info in patient_info.values()]):.2f}")


In [ ]:
def show_trajectory_examples(patient_trajectories, patient_info, num_examples=3):
    """显示轨迹示例"""
    
    print("="*80)
    print("Examples of patient trajectories")
    print("="*80)
    
    for i, (subject_id, trajectory) in enumerate(list(patient_trajectories.items())[:num_examples]):
        info = patient_info[subject_id]
        print(f"\nPatient {subject_id}:")
        print(f"  Number of admissions: {info['num_admissions']}")
        print(f"  Number of total diagnoses: {info['total_diagnoses']}")
        print(f"  Admission time: {[date.strftime('%Y-%m-%d-%H-%M-%S') for date in info['admission_dates']]}")
        print(f"  Discharge time: {[date.strftime('%Y-%m-%d-%H-%M-%S') for date in info['discharge_dates']]}")
        print(f"  Trajectory:")
        
        for j, visit in enumerate(trajectory):
            print(f"    Admission {j+1}: {visit[:10]}{'...' if len(visit) > 10 else ''} ({len(visit)} diagnoses)")

show_trajectory_examples(patient_trajectories, patient_info)

In [ ]:
from itertools import combinations
import collections

def extract_disease_pairs_from_trajectories(patient_trajectories):
    """
    Extract disease pairs from patient trajectories
    
    Args:
        patient_trajectories: dict, key is subject_id, value is a list of trajectories

    Returns:
        all_pairs: set, all unique disease pairs
        pair_counts: dict, count of each disease pair
        visit_pairs: dict, each patient's each visit's disease pairs
    """

    all_pairs = set()
    pair_counts = collections.defaultdict(int)
    visit_pairs = {}

    for subject_id, trajectory in patient_trajectories.items():
        visit_pairs[subject_id] = []
        
        for visit_idx, visit_codes in enumerate(trajectory):
            # 获取该次visit的所有unique codes
            unique_codes = list(set(visit_codes))

            # 生成该次visit的所有disease pairs
            visit_pairs_list = []
            for code1, code2 in combinations(unique_codes, 2):
                pair = tuple(sorted([code1, code2]))
                visit_pairs_list.append(pair)
                all_pairs.add(pair)
                pair_counts[pair] += 1
                
            visit_pairs[subject_id].append(visit_pairs_list)

    return all_pairs, dict(pair_counts), visit_pairs

In [ ]:
all_pairs, pair_counts, visit_pairs = extract_disease_pairs_from_trajectories(patient_trajectories)

print(f"\n=== Disease Code Pairs Statistics ===")
print(f"Number of unique pairs: {len(all_pairs):,}")
print(f"Total pair occurrences: {sum(pair_counts.values()):,}")
print(f"Average pair occurrences: {sum(pair_counts.values()) / len(all_pairs):.2f}")

# 显示最常见的pairs
print(f"\nMost common 10 disease code pairs:")
sorted_pairs = sorted(pair_counts.items(), key=lambda x: x[1], reverse=True)
for i, (pair, count) in enumerate(sorted_pairs[:10]):
    print(f"  {i+1}. {pair[0]} - {pair[1]}: {count:,} occurrences")

### DTW

In [ ]:
types = diagnoses_clean['ICD9_CODE'].unique()
types

In [ ]:
def create_types_dict(icd9_codes):
    """
    Create a dictionary mapping ICD-9 codes to indices
    """
    types_dict = {}

    sorted_codes = sorted(icd9_codes)

    for idx, code in enumerate(sorted_codes):
        types_dict[code] = idx

    return types_dict

types_dict = create_types_dict(types)

In [ ]:
def prepare_trajectories_for_dtw(patient_trajectories, patient_info, types_dict, min_visits=3):
    """
    Prepare trajectories for DTW calculation
    
    Args:
        patient_trajectories: dict, key is subject_id, value is a list of trajectories
        patient_info: dict, contains information of patients
        min_visits: int, minimum number of visits

    Returns:
        dtw_ready_trajectories: dict, key is subject_id, value is a list of visit vectors (multi-hot vector)
    """
    multi_visit_patients = {
        pid: trajectory for pid, trajectory in patient_trajectories.items()
        if patient_info[pid]['num_admissions'] >= min_visits
    }

    print(f"Number of patients with at least {min_visits} visits: {len(multi_visit_patients)}")

    dtw_ready_trajectories = {}

    for pid, trajectory in multi_visit_patients.items():
        # multi-hot vector
        visit_vectors = []
        for visit in trajectory:
            visit_vector = np.zeros(len(types_dict))
            for code in visit:
                if code in types_dict:
                    visit_vector[types_dict[code]] = 1
            visit_vectors.append(visit_vector)

        dtw_ready_trajectories[pid] = visit_vectors

    return dtw_ready_trajectories

In [ ]:
dtw_trajectories = prepare_trajectories_for_dtw(patient_trajectories, patient_info, types_dict)

In [ ]:
dtw_trajectories[36]

In [ ]:
dtw_trajectories.keys()

In [ ]:
from dtaidistance import dtw
import matplotlib.pyplot as plt
import seaborn as sns

# def dtw_with_jaccard(dtw_ready_trajectories, sample_size=100):
#     """
#     Calculate DTW with Jaccard distance
#     """

#     def jaccard_distance(vec1, vec2):
#         """计算两个multi-hot向量的Jaccard距离"""
#         intersection = np.sum(vec1 & vec2)
#         union = np.sum(vec1 | vec2)
#         if union == 0:
#             return 0
#         return 1 - (intersection / union)

#     # 采样
#     patient_ids = list(dtw_ready_trajectories.keys())[:sample_size]
#     n_patients = len(patient_ids)
#     dtw_distance_matrix = np.zeros((n_patients, n_patients))

#     print(f"Calculating DTW matrix with Jaccard distance ({n_patients} x {n_patients})...")
    
#     for i, pid1 in enumerate(patient_ids):
#         for j, pid2 in enumerate(patient_ids):
#             if i <= j:
#                 trajectory1 = dtw_ready_trajectories[pid1]
#                 trajectory2 = dtw_ready_trajectories[pid2]

#                 distance = dtw.distance(trajectory1, trajectory2, use_c=False)
#                 dtw_distance_matrix[i, j] = distance
#                 dtw_distance_matrix[j, i] = distance

#     return dtw_distance_matrix, patient_ids

In [ ]:
def manual_dtw_with_jaccard(trajectory1, trajectory2):
    """
    手动实现DTW算法，使用Jaccard距离作为局部代价
    """
    def jaccard_distance(vec1, vec2):
        xb = vec1.astype(bool)
        yb = vec2.astype(bool)
        inter = np.count_nonzero(xb & yb)
        union = np.count_nonzero(xb | yb)
        if union == 0:
            return 0.0
        return 1.0 - inter / union
    
    n, m = len(trajectory1), len(trajectory2)
    
    # 初始化DP矩阵
    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0.0
    
    # 填充DP矩阵，DTW核心算法部分
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = jaccard_distance(trajectory1[i-1], trajectory2[j-1])
            # 递推公式
            dp[i, j] = cost + min(
                dp[i-1, j],     # 垂直移动：重复使用trajectory1的i-1时刻
                dp[i, j-1],     # 水平移动：重复使用trajectory2的j-1时刻
                dp[i-1, j-1])   # 对角移动：正常的一对一对齐
    
    return dp[n, m]

def dtw_with_jaccard_manual(dtw_ready_trajectories, sample_size=500):
    """
    使用手动DTW实现计算Jaccard距离
    """
    patient_ids = list(dtw_ready_trajectories.keys())[:sample_size]
    n_patients = len(patient_ids)
    dtw_distance_matrix = np.zeros((n_patients, n_patients))
    
    print(f"Calculating DTW matrix with manual Jaccard distance ({n_patients} x {n_patients})...")
    
    for i, pid1 in enumerate(patient_ids):
        for j, pid2 in enumerate(patient_ids):
            if i <= j:
                trajectory1 = dtw_ready_trajectories[pid1]
                trajectory2 = dtw_ready_trajectories[pid2]
                
                distance = manual_dtw_with_jaccard(trajectory1, trajectory2)
                dtw_distance_matrix[i, j] = distance
                dtw_distance_matrix[j, i] = distance
    
    return dtw_distance_matrix, patient_ids

#### 可视化DTW算法

In [ ]:
def visualize_dtw_alignment(trajectory1, trajectory2, patient_id1, patient_id2):
    """
    可视化DTW对齐路径和轨迹
    """
    def jaccard_distance(vec1, vec2):
        xb = vec1.astype(bool)
        yb = vec2.astype(bool)
        inter = np.count_nonzero(xb & yb)
        union = np.count_nonzero(xb | yb)
        if union == 0:
            return 0.0
        return 1.0 - inter / union
    
    n, m = len(trajectory1), len(trajectory2)
    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0.0
    
    # 计算DP矩阵
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = jaccard_distance(trajectory1[i-1], trajectory2[j-1])
            dp[i, j] = cost + min(dp[i-1, j], dp[i, j-1], dp[i-1, j-1])
    
    # 回溯找到最优路径
    path = []
    i, j = n, m
    while i > 0 and j > 0:
        path.append((i-1, j-1))
        if dp[i-1, j] <= dp[i, j-1] and dp[i-1, j] <= dp[i-1, j-1]:
            i -= 1
        elif dp[i, j-1] <= dp[i-1, j-1]:
            j -= 1
        else:
            i -= 1
            j -= 1
    
    path = path[::-1]
    
    # 创建可视化
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    
    # 1. DTW路径热力图
    ax1.imshow(dp[1:, 1:], cmap='viridis', aspect='auto')
    ax1.set_title(f'DTW Cost Matrix\nPatient {patient_id1} vs Patient{patient_id2}')
    ax1.set_xlabel('Patient 2 Trajectory Time Points')
    ax1.set_ylabel('Patient 1 Trajectory Time Points')
    
    # 绘制最优路径
    path_x = [p[1] for p in path]
    path_y = [p[0] for p in path]
    ax1.plot(path_x, path_y, 'r-', linewidth=3, alpha=0.8)
    ax1.scatter(path_x, path_y, c='red', s=50, zorder=5)
    
    # 2. 轨迹对比图
    # 患者1轨迹
    traj1_diagnoses = [np.sum(visit) for visit in trajectory1]
    ax2.plot(range(len(traj1_diagnoses)), traj1_diagnoses, 'bo-', 
             label=f'Patient {patient_id1}', linewidth=2, markersize=8)
    
    # 患者2轨迹
    traj2_diagnoses = [np.sum(visit) for visit in trajectory2]
    ax2.plot(range(len(traj2_diagnoses)), traj2_diagnoses, 'ro-', 
             label=f'Patient {patient_id2}', linewidth=2, markersize=8)
    
    ax2.set_title('Patient Trajectory Comparison')
    ax2.set_xlabel('Number of Admissions')
    ax2.set_ylabel('Number of Diagnoses')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. 对齐路径可视化
    ax3.set_xlim(-0.5, max(n, m) - 0.5)
    ax3.set_ylim(-0.5, 1.5)
    
    # 绘制对齐线
    for i, (t1_idx, t2_idx) in enumerate(path):
        ax3.plot([t1_idx, t2_idx], [0, 1], 'b-', alpha=0.6, linewidth=1)
        ax3.scatter([t1_idx, t2_idx], [0, 1], c=['blue', 'red'], s=30, zorder=5)
    
    ax3.set_title('DTW Alignment Path')
    ax3.set_xlabel('Time Points')
    ax3.set_ylabel('Patient')
    ax3.set_yticks([0, 1])
    ax3.set_yticklabels([f'Patient {patient_id1}', f'Patient {patient_id2}'])
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"DTW Distance: {dp[n, m]:.4f}")
    print(f"Alignment Path Length: {len(path)}")
    print(f"Patient 1 Trajectory Length: {n}, Patient 2 Trajectory Length: {m}")
    
    return path, dp[n, m]

# 使用示例
patient_ids = list(dtw_trajectories.keys())
path, distance = visualize_dtw_alignment(
    dtw_trajectories[patient_ids[0]], 
    dtw_trajectories[patient_ids[6]], 
    patient_ids[0], 
    patient_ids[6]
)

#### 计算DTW距离矩阵

In [ ]:
D, patient_ids = dtw_with_jaccard_manual(dtw_trajectories, sample_size=len(dtw_trajectories))

plt.figure(figsize=(10, 8))
sns.heatmap(D, cmap='viridis', square=True, cbar_kws={'label': 'DTW Distance'})
plt.title('DTW Distance Matrix (Jaccard)')
plt.xlabel('Patients')
plt.ylabel('Patients')
plt.show()

# 距离分布
tri = D[np.triu_indices_from(D, k=1)]
plt.figure(figsize=(8, 6))
plt.hist(tri, bins=30, color='#2E86AB', alpha=0.8)
plt.xlabel('Pairwise DTW Distance')
plt.ylabel('Count')
plt.title('DTW Distance Distribution')
plt.show()

print(f"DTW Distance Statistics:")
print(f"  Average: {tri.mean():.4f}")
print(f"  Median: {np.median(tri):.4f}")
print(f"  Min: {tri.min():.4f}")
print(f"  Max: {tri.max():.4f}")

In [ ]:
# 查看最相似的患者对
def find_most_similar_pairs(D, patient_ids, top_k=10):
    """找到最相似的患者对"""
    n = len(patient_ids)
    pairs = []
    
    for i in range(n):
        for j in range(i+1, n):
            pairs.append((patient_ids[i], patient_ids[j], D[i, j]))
    
    # 按距离排序
    pairs.sort(key=lambda x: x[2])
    
    print(f"The most similar {top_k} pairs of patients:")
    for i, (pid1, pid2, dist) in enumerate(pairs[:top_k]):
        print(f"  {i+1}. Patient {pid1} vs Patient {pid2}: Distance = {dist:.4f}")
    
    return pairs[:top_k]

similar_pairs = find_most_similar_pairs(D, patient_ids)

In [ ]:
patient_trajectories[28182]

In [ ]:
patient_trajectories[71527]

#### Clustering

In [ ]:
def incremental_clustering(D, patient_ids, threshold):
    """
    基于距离阈值的增量聚类算法
    
    Args:
        D: DTW距离矩阵 (n x n)
        patient_ids: 患者ID列表
        threshold: 距离阈值
    
    Returns:
        clusters: dict, key是cluster_id, value是患者ID列表
        cluster_assignments: dict, key是patient_id, value是cluster_id
    """
    n = len(patient_ids)
    clusters = {}  # {cluster_id: [patient_ids]}
    cluster_assignments = {}  # {patient_id: cluster_id}
    next_cluster_id = 0
    
    # 创建患者ID到索引的映射
    pid_to_idx = {pid: i for i, pid in enumerate(patient_ids)}
    
    print(f"Start clustering with threshold: {threshold}")
    print(f"Total number of patients: {n}")
    
    for i, current_pid in enumerate(patient_ids):
        current_idx = pid_to_idx[current_pid]
        
        if i == 0:
            # 第一个患者自动创建第一个cluster
            clusters[next_cluster_id] = [current_pid]
            cluster_assignments[current_pid] = next_cluster_id
            next_cluster_id += 1
            print(f"Patient {current_pid} -> Create newcluster {next_cluster_id-1}")
            continue
        
        # 计算与所有已分配患者的距离
        min_distance = float('inf')
        best_cluster = None
        
        for cluster_id, cluster_patients in clusters.items():
            # 计算与当前cluster中所有患者的平均距离
            distances = []
            for pid in cluster_patients:
                pid_idx = pid_to_idx[pid]
                distances.append(D[current_idx, pid_idx])
            
            # 使用最小距离作为与cluster的距离
            cluster_distance = min(distances)
            
            if cluster_distance < min_distance:
                min_distance = cluster_distance
                best_cluster = cluster_id
        
        # 决定是否加入现有cluster或创建新cluster
        if min_distance <= threshold:
            # 加入最近的cluster
            clusters[best_cluster].append(current_pid)
            cluster_assignments[current_pid] = best_cluster
            print(f"Patient {current_pid} -> Join cluster {best_cluster} (distance: {min_distance:.4f})")
        else:
            # 创建新cluster
            clusters[next_cluster_id] = [current_pid]
            cluster_assignments[current_pid] = next_cluster_id
            print(f"Patient {current_pid} -> Create new cluster {next_cluster_id} (minimum distance: {min_distance:.4f})")
            next_cluster_id += 1
    
    return clusters, cluster_assignments

def analyze_clustering_results(clusters, cluster_assignments):
    """分析聚类结果"""
    print(f"\n=== Clustering Results Analysis ===")
    print(f"Total number of clusters: {len(clusters)}")
    print(f"Total number of patients: {len(cluster_assignments)}")
    
    cluster_sizes = [len(cluster) for cluster in clusters.values()]
    print(f"Cluster size statistics:")
    print(f"  Average: {np.mean(cluster_sizes):.2f}")
    print(f"  Median: {np.median(cluster_sizes):.2f}")
    print(f"  Maximum: {max(cluster_sizes)}")
    print(f"  Minimum: {min(cluster_sizes)}")
    
    # 显示每个cluster的信息
    print(f"\nEach cluster details:")
    for cluster_id, patients in clusters.items():
        print(f"  Cluster {cluster_id}: {len(patients)} patients")
        if len(patients) <= 10:  # 只显示小cluster的详细信息
            print(f"    Patients: {patients}")
    
    return cluster_sizes

def visualize_clustering_results(clusters, cluster_assignments, D, patient_ids):
    """可视化聚类结果"""
    # 创建cluster标签
    cluster_labels = []
    for pid in patient_ids:
        cluster_labels.append(cluster_assignments[pid])
    
    # 重新排列距离矩阵以显示cluster结构
    sorted_indices = np.argsort(cluster_labels)
    sorted_D = D[np.ix_(sorted_indices, sorted_indices)]
    sorted_labels = [cluster_labels[i] for i in sorted_indices]
    
    # 绘制热力图
    plt.figure(figsize=(12, 10))
    sns.heatmap(sorted_D, cmap='viridis', square=True, 
                cbar_kws={'label': 'DTW Distance'})
    plt.title('DTW Distance Matrix with Clustering Results')
    plt.xlabel('Patients (sorted by cluster)')
    plt.ylabel('Patients (sorted by cluster)')
    
    # 添加cluster边界线
    unique_labels = sorted(set(cluster_labels))
    boundaries = []
    current_pos = 0
    for label in unique_labels:
        count = sorted_labels.count(label)
        current_pos += count
        boundaries.append(current_pos)
    
    for boundary in boundaries[:-1]:
        plt.axhline(y=boundary, color='red', linewidth=0.2, alpha=0.8)
        plt.axvline(x=boundary, color='red', linewidth=0.2, alpha=0.8)
    
    plt.show()
    
    # 绘制cluster大小分布
    # cluster_sizes = [len(cluster) for cluster in clusters.values()]
    # plt.figure(figsize=(10, 6))
    # plt.hist(cluster_sizes, bins=20, color='skyblue', alpha=0.7, edgecolor='black')
    # plt.xlabel('Cluster Size')
    # plt.ylabel('Number of Clusters')
    # plt.title('Distribution of Cluster Sizes')
    # plt.grid(True, alpha=0.3)
    # plt.show()

# 尝试不同的阈值
thresholds = [2.5, 2.53, 2.55, 2.6, 2.62, 2.65, 2.7, 2.75, 2.8]

for threshold in thresholds:
    print(f"\n{'='*60}")
    print(f"Test threshold: {threshold}")
    print(f"{'='*60}")
    
    clusters, cluster_assignments = incremental_clustering(D, patient_ids, threshold)
    cluster_sizes = analyze_clustering_results(clusters, cluster_assignments)
    
    # 只对第一个阈值进行可视化
    # if threshold == thresholds[0]:
    visualize_clustering_results(clusters, cluster_assignments, D, patient_ids)